# Get citing & cited opinions metadata for Federal Courts

In the previous experiments, we've primarily focused on SCOTUS. This is to expand the dataset to lower courts in the Federal jurisdiction, including the Federal circuit courts and the Federal district courts. 

This notebook documents the steps I undertook to:
1. Identify the Federal jurisdiction courts to include
2. Use Django Shell to sample target cases in the target courts from CL Replica
3. Use Django Shell to get all cases that cited the sampled target cases - these are the citing cases
4. Use Django Shell to get all authorities for the citing cases - these are the cited cases, including ones in the specified courts (target cases) and the ones not in the specified courts
5. Use Django Shell to get related metadata for the citing and cited cases

# Import Libaries

In [1]:
import json

import numpy as np
import pandas as pd

# Load Court Hierarchy data

In [2]:
df = pd.read_csv("../experiments_624/court_hierarchy_flp.csv")
df.head()

,id,full_name,jurisdiction,jurisdiction_name,jurisdiction_type,jurisdiction_state,appeals_to_full_name,appeals_to_id,note
0,scotus,Supreme Court of the United States,F,Federal Appellate,Federal,Federal,NaN,NaN,NaN
1,cafc,Court of Appeals for the Federal Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN
2,ca1,Court of Appeals for the First Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN
3,ca2,Court of Appeals for the Second Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN
4,ca3,Court of Appeals for the Third Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN


In [3]:
df["jurisdiction_name"].value_counts()

jurisdiction_name
Federal Bankruptcy          141
Federal District             94
State Appellate              57
State Supreme                52
Federal Appellate            14
Federal Bankruptcy Panel      8
State Trial                   6
Territory Supreme             5
Territory Trial               4
Territory Appellate           2
Name: count, dtype: int64

In [4]:
fed_appellate = df[df["jurisdiction_name"] == "Federal Appellate"]
fed_appellate["full_name"]

0            Supreme Court of the United States
1      Court of Appeals for the Federal Circuit
2        Court of Appeals for the First Circuit
3       Court of Appeals for the Second Circuit
4        Court of Appeals for the Third Circuit
5       Court of Appeals for the Fourth Circuit
6        Court of Appeals for the Fifth Circuit
7        Court of Appeals for the Sixth Circuit
8      Court of Appeals for the Seventh Circuit
9       Court of Appeals for the Eighth Circuit
10       Court of Appeals for the Ninth Circuit
11       Court of Appeals for the Tenth Circuit
12    Court of Appeals for the Eleventh Circuit
13        Court of Appeals for the D.C. Circuit
Name: full_name, dtype: object

In [5]:
fed_district = df[df["jurisdiction_name"] == "Federal District"]
fed_district["full_name"]

163        District Court, District of Columbia
164                District Court, M.D. Alabama
165                District Court, N.D. Alabama
166                District Court, S.D. Alabama
167                   District Court, D. Alaska
                         ...                   
252                  District Court, D. Wyoming
253                     District Court, D. Guam
254    District Court, Northern Mariana Islands
255              District Court, D. Puerto Rico
256              District Court, Virgin Islands
Name: full_name, Length: 94, dtype: object

In [6]:
fed_district["jurisdiction_state"].value_counts()

jurisdiction_state
New York                    4
Texas                       4
California                  4
North Carolina              3
Florida                     3
Tennessee                   3
Illinois                    3
Georgia                     3
West Virginia               3
Alabama                     3
Pennsylvania                3
Oklahoma                    3
Louisiana                   3
Ohio                        2
Missouri                    2
Mississippi                 2
Michigan                    2
Kentucky                    2
Iowa                        2
Indiana                     2
Virginia                    2
Washington                  2
Wisconsin                   2
Arkansas                    2
Vermont                     1
Utah                        1
South Dakota                1
South Carolina              1
Wyoming                     1
Rhode Island                1
Oregon                      1
Guam                        1
Northern Mariana Isla

In [7]:
court_ids = fed_appellate["id"].to_list() + fed_district["id"].to_list()
len(court_ids)

108

# Use Django Shell to get the list of target case cluster ids and citing case cluster ids

In [8]:
with open("data/citing_ids.txt", "r") as f:
    citing_ids = [line.strip() for line in f]
len(citing_ids)

1352

In [9]:
with open("data/target_ids.txt", "r") as f:
    target_ids = [int(line.strip()) for line in f]
len(target_ids)

193

## Ensure the citing cases are not already part of the SCOTUS set in review

In [10]:
scotus = pd.read_json("../experiments_624/data/scotus_citing_cited_sampled.json")
len(scotus)

21925

In [11]:
scotus_citing = list(set(scotus["citing_cluster_id"].to_list()))
len(scotus_citing)

482

In [12]:
assert len(set(citing_ids) - set(scotus_citing)) == len(citing_ids)

# Use Django Shell to get the citing cases metadata

In [13]:
with open('data/citing_opinions.json', 'r') as f:
    results = json.load(f)

In [14]:
records = []
for case_id, case_data in results.items():
    record = {'citing_cluster_id': int(case_id)}
    record.update({k: v for k, v in case_data.items()})
    
    opinion_filenames = [op['opinion_filename'] for op in case_data.get('opinion_data', [])]
    record['opinion_filenames'] = opinion_filenames
    
    records.append(record)

citing_df = pd.DataFrame(records)
citing_df.head()

,citing_cluster_id,citing_url,citing_court_id,citing_court_name,opinion_data,cited_cluster_ids,opinion_filenames
0,237668,https://www.courtlistener.com/opinion/237668/t...,ca6,Court of Appeals for the Sixth Circuit,"[{'opinion_id': 237668, 'opinion_api': None, '...","[1569044, 4025274, 4025744, 4026202, 4026841, ...",[237668_010combined.txt]
1,287111,https://www.courtlistener.com/opinion/287111/t...,ca4,Court of Appeals for the Fourth Circuit,"[{'opinion_id': 287111, 'opinion_api': None, '...","[272551, 272552, 8892883, 8893769, 8895154, 89...",[287111_010combined.txt]
2,311491,https://www.courtlistener.com/opinion/311491/u...,ca8,Court of Appeals for the Eighth Circuit,"[{'opinion_id': 311491, 'opinion_api': None, '...","[91715, 102106, 102166, 225722, 231235, 237946...","[311491_010combined.txt, 311491_035concurrence..."
3,319907,https://www.courtlistener.com/opinion/319907/u...,ca5,Court of Appeals for the Fifth Circuit,"[{'opinion_id': 319907, 'opinion_api': None, '...","[270201, 272454, 273766, 277868, 285695, 28679...",[319907_010combined.txt]
4,352343,https://www.courtlistener.com/opinion/352343/f...,ca9,Court of Appeals for the Ninth Circuit,"[{'opinion_id': 352343, 'opinion_api': None, '...","[99426, 103923, 106452, 303417, 311364, 891612...",[352343_010combined.txt]


## Get all cited case ids to a list for extracting the metadata

In [15]:
#with open('data/cited_ids.txt', 'w') as f:
#    for each in list(set(citing_df["cited_cluster_ids"].explode().dropna().tolist())):
#        f.write(f"{each}\n")

# Use Django Shell to extract the metadata for the cited cases

In [16]:
with open('data/cited_opinions.json', 'r') as f:
    results = json.load(f)

In [17]:
cited_metadata = pd.DataFrame.from_dict(results, orient='index')
cited_metadata = cited_metadata.reset_index().rename(columns={'index': 'cited_cluster_id'})
cited_metadata["cited_cluster_id"] = cited_metadata["cited_cluster_id"].astype(int)
cited_metadata.head()

,cited_cluster_id,cited_url,cited_court_id,cited_court_name,cited_case_name_short,cited_case_name,cited_case_name_full,cited_citations
0,39,https://www.courtlistener.com/opinion/39/unite...,ca2,Court of Appeals for the Second Circuit,Awad,United States v. Awad,"UNITED STATES of America, Appellee-Cross-Appel...","[2010 WL 840243, 2010 U.S. App. LEXIS 5128, 59..."
1,65,https://www.courtlistener.com/opinion/65/unite...,ca3,Court of Appeals for the Third Circuit,,United States v. Paul Shenandoah,"UNITED STATES of America v. Paul SHENANDOAH, A...","[2010 WL 431897, 2010 U.S. App. LEXIS 2652, 59..."
2,85,https://www.courtlistener.com/opinion/85/morto...,ca9,Court of Appeals for the Ninth Circuit,Morton,Morton v. Hall,"Bruce Alan MORTON, Plaintiff-Appellant, v. Jam...","[2010 WL 843879, 2010 U.S. App. LEXIS 5209, 59..."
3,152,https://www.courtlistener.com/opinion/152/unit...,ca11,Court of Appeals for the Eleventh Circuit,Frank,United States v. Frank,"UNITED STATES of America, Plaintiff-Appellee, ...","[2010 WL 890451, 599 F.3d 1221]"
4,254,https://www.courtlistener.com/opinion/254/calv...,ca1,Court of Appeals for the First Circuit,Calvao,Calvao v. Town of Framingham,"Duarte CALVAO, Et Al., Plaintiffs, Appellants,...","[2010 WL 936553, 599 F.3d 10]"


In [18]:
len(cited_metadata)

22923

# Create result_df by merging citing and cited metadatas

In [19]:
result_df = citing_df.explode("cited_cluster_ids").reset_index(drop=True)
len(result_df)

41613

In [20]:
result_df = result_df.rename(columns={"cited_cluster_ids": "cited_cluster_id"})
result_df.head()

,citing_cluster_id,citing_url,citing_court_id,citing_court_name,opinion_data,cited_cluster_id,opinion_filenames
0,237668,https://www.courtlistener.com/opinion/237668/t...,ca6,Court of Appeals for the Sixth Circuit,"[{'opinion_id': 237668, 'opinion_api': None, '...",1569044,[237668_010combined.txt]
1,237668,https://www.courtlistener.com/opinion/237668/t...,ca6,Court of Appeals for the Sixth Circuit,"[{'opinion_id': 237668, 'opinion_api': None, '...",4025274,[237668_010combined.txt]
2,237668,https://www.courtlistener.com/opinion/237668/t...,ca6,Court of Appeals for the Sixth Circuit,"[{'opinion_id': 237668, 'opinion_api': None, '...",4025744,[237668_010combined.txt]
3,237668,https://www.courtlistener.com/opinion/237668/t...,ca6,Court of Appeals for the Sixth Circuit,"[{'opinion_id': 237668, 'opinion_api': None, '...",4026202,[237668_010combined.txt]
4,237668,https://www.courtlistener.com/opinion/237668/t...,ca6,Court of Appeals for the Sixth Circuit,"[{'opinion_id': 237668, 'opinion_api': None, '...",4026841,[237668_010combined.txt]


In [21]:
result_df = result_df.merge(cited_metadata, how="left", on="cited_cluster_id")
len(result_df)

41613

In [22]:
result_df.columns

Index(['citing_cluster_id', 'citing_url', 'citing_court_id',
       'citing_court_name', 'opinion_data', 'cited_cluster_id',
       'opinion_filenames', 'cited_url', 'cited_court_id', 'cited_court_name',
       'cited_case_name_short', 'cited_case_name', 'cited_case_name_full',
       'cited_citations'],
      dtype='object')

In [23]:
result_df = result_df[['citing_cluster_id', 'citing_url', 'citing_court_id',
       'citing_court_name', 'opinion_data', 'opinion_filenames', 
       'cited_cluster_id', 'cited_url', 'cited_court_id', 'cited_court_name',
       'cited_case_name_short', 'cited_case_name', 'cited_case_name_full',
       'cited_citations']]

In [24]:
result_df.head()

,citing_cluster_id,citing_url,citing_court_id,citing_court_name,opinion_data,opinion_filenames,cited_cluster_id,cited_url,cited_court_id,cited_court_name,cited_case_name_short,cited_case_name,cited_case_name_full,cited_citations
0,237668,https://www.courtlistener.com/opinion/237668/t...,ca6,Court of Appeals for the Sixth Circuit,"[{'opinion_id': 237668, 'opinion_api': None, '...",[237668_010combined.txt],1569044,https://www.courtlistener.com/opinion/1569044/...,ohsd,"District Court, S.D. Ohio",Shafer,London Guarantee & Accident Co. v. Shafer,"LONDON GUARANTEE & ACCIDENT CO., Limited, v. S...","[1940 U.S. Dist. LEXIS 2322, 35 F. Supp. 647]"
1,237668,https://www.courtlistener.com/opinion/237668/t...,ca6,Court of Appeals for the Sixth Circuit,"[{'opinion_id': 237668, 'opinion_api': None, '...",[237668_010combined.txt],4025274,https://www.courtlistener.com/opinion/4025274/...,ohio,Ohio Supreme Court,,Bloom-Rosenblum-Kline Co. v. Union Indemnity Co.,The Bloom-Rosenblum-Kline Co. v. Union Indemni...,"[1929 Ohio LEXIS 287, 7 Ohio Law. Abs. 379, 12..."
2,237668,https://www.courtlistener.com/opinion/237668/t...,ca6,Court of Appeals for the Sixth Circuit,"[{'opinion_id': 237668, 'opinion_api': None, '...",[237668_010combined.txt],4025744,https://www.courtlistener.com/opinion/4025744/...,ohio,Ohio Supreme Court,Leonard,Leonard v. Murdock,"Leonard, Appellee, v. Murdock Et Al.; The Ocea...","[1946 Ohio LEXIS 268, 33 Ohio Op. 269, 147 Ohi..."
3,237668,https://www.courtlistener.com/opinion/237668/t...,ca6,Court of Appeals for the Sixth Circuit,"[{'opinion_id': 237668, 'opinion_api': None, '...",[237668_010combined.txt],4026202,https://www.courtlistener.com/opinion/4026202/...,ohio,Ohio Supreme Court,,"Mitchell v. Great Eastern Stages, Inc.","Mitchell, Admx., Appellee, v. Great Eastern St...","[1942 Ohio LEXIS 417, 141 A.L.R. 624, 23 Ohio ..."
4,237668,https://www.courtlistener.com/opinion/237668/t...,ca6,Court of Appeals for the Sixth Circuit,"[{'opinion_id': 237668, 'opinion_api': None, '...",[237668_010combined.txt],4026841,https://www.courtlistener.com/opinion/4026841/...,ohio,Ohio Supreme Court,Bobier,Bobier v. National Casualty Co.,"Bobier, D. B. A. Federal Appliance Service Co....","[1944 Ohio LEXIS 400, 28 Ohio Op. 138, 143 Ohi..."


## Tag the target cases from the target courts

In [25]:
result_df.loc[result_df["cited_cluster_id"].isin(target_ids), "cited_target"] = 1
result_df.loc[~result_df["cited_cluster_id"].isin(target_ids), "cited_target"] = 0

## Do some EDA

In [26]:
eda_cols = ['citing_cluster_id', 'citing_court_name', 'cited_cluster_id', 'cited_court_name', 'cited_target']

for col in eda_cols:
    print("----------")
    print(result_df[col].nunique())
    display(result_df[col].value_counts())

----------
1352


citing_cluster_id
9739659    250
1514994    206
443256     195
8935676    193
1639439    189
          ... 
560249       1
4865932      1
7346086      1
8614269      1
8162245      1
Name: count, Length: 1352, dtype: int64

----------
171


citing_court_name
Court of Appeals for the Ninth Circuit               3890
District Court, S.D. Florida                         2285
District Court, N.D. California                      2081
Court of Appeals for the Fifth Circuit               2043
District Court, E.D. California                      1804
                                                     ... 
Superior Court of Delaware                              5
United States Bankruptcy Court, M.D. Pennsylvania       5
Supreme Court of Delaware                               5
United States Bankruptcy Court, E.D. Arkansas           4
Virginia Beach County Circuit Court                     3
Name: count, Length: 171, dtype: int64

----------
22923


cited_cluster_id
145730     157
145875     137
347621     133
2411022    117
111719     114
          ... 
362491       1
417785       1
1479016      1
1798966      1
4236912      1
Name: count, Length: 22923, dtype: int64

----------
338


cited_court_name
Supreme Court of the United States                         10173
Court of Appeals for the Ninth Circuit                      5559
Court of Appeals for the Fifth Circuit                      2463
Court of Appeals for the Eleventh Circuit                   2170
Court of Appeals for the Second Circuit                     2055
                                                           ...  
Court of Appeals for the Armed Forces                          1
Pennsylvania Court of Common Pleas, Columbia County            1
Pennsylvania Court of Common Pleas, Philadelphia County        1
U.S. Circuit Court for the District of Indiana                 1
U.S. Navy-Marine Corps Court of Military Review                1
Name: count, Length: 338, dtype: int64

----------
2


cited_target
0.0    40255
1.0     1358
Name: count, dtype: int64

In [27]:
df_target = result_df[result_df["cited_target"] == 1]
print(df_target["cited_court_name"].nunique())
df_target["cited_court_name"].value_counts()

36


cited_court_name
Court of Appeals for the Ninth Circuit       321
Court of Appeals for the Fourth Circuit      166
District Court, S.D. Georgia                 117
Court of Appeals for the Fifth Circuit        92
Court of Appeals for the Seventh Circuit      71
District Court, N.D. California               69
District Court, C.D. California               67
District Court, S.D. Florida                  61
Court of Appeals for the Second Circuit       55
District Court, S.D. New York                 52
Court of Appeals for the Eighth Circuit       38
District Court, E.D. New York                 27
Court of Appeals for the Third Circuit        27
District Court, E.D. Pennsylvania             27
Court of Appeals for the Eleventh Circuit     27
Court of Appeals for the Sixth Circuit        25
District Court, E.D. Wisconsin                21
District Court, W.D. Michigan                 16
District Court, D. Hawaii                     12
District Court, M.D. Alabama                  10
Dis

# Save the data for future use

In [28]:
result_df.to_json("data/fed_citing_cited.json")